# dto-fine-tune — training + inference

Runs the pipeline from https://github.com/Igufu/dto-fine-tune.

In [ ]:
!git clone https://github.com/Igufu/dto-fine-tune.git
%cd dto-fine-tune

In [ ]:
!pip install -q -r requirements.txt 'accelerate>=1.1.0' matplotlib

In [ ]:
!python build_datasets.py

In [ ]:
!python train.py

In [ ]:
!python inference.py

In [ ]:
!head -40 predictions.ann

## Training plots
Reads `models/<class>/log_history.json` and `models/<class>/val_counts.json` written by `train.py`.

In [ ]:
import json
import matplotlib.pyplot as plt

CLASSES = ["deontic", "object", "attribute"]
best_f1_by_class = {}

for name in CLASSES:
    with open(f"models/{name}/log_history.json") as f:
        log = json.load(f)
    with open(f"models/{name}/val_counts.json") as f:
        counts = json.load(f)

    train_rows = [r for r in log if "loss" in r and "eval_loss" not in r]
    eval_rows = [r for r in log if "eval_f1" in r]

    tr_epoch = [r["epoch"] for r in train_rows]
    tr_loss = [r["loss"] for r in train_rows]
    ev_epoch = [r["epoch"] for r in eval_rows]
    ev_loss = [r["eval_loss"] for r in eval_rows]
    ev_p = [r["eval_precision"] for r in eval_rows]
    ev_r = [r["eval_recall"] for r in eval_rows]
    ev_f = [r["eval_f1"] for r in eval_rows]

    best_i = max(range(len(ev_f)), key=lambda i: ev_f[i]) if ev_f else None
    best_f1_by_class[name] = ev_f[best_i] if best_i is not None else 0.0

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(name, fontsize=14)

    ax = axes[0]
    ax.plot(tr_epoch, tr_loss, label="train loss", marker="o")
    ax.plot(ev_epoch, ev_loss, label="eval loss", marker="s")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_title("loss")
    ax.legend(); ax.grid(True, alpha=0.3)

    ax = axes[1]
    ax.plot(ev_epoch, ev_p, label="precision", marker="o")
    ax.plot(ev_epoch, ev_r, label="recall", marker="s")
    ax.plot(ev_epoch, ev_f, label="f1", marker="^")
    if best_i is not None:
        ax.scatter([ev_epoch[best_i]], [ev_f[best_i]], color="red", s=120, zorder=5,
                   label=f"best f1 (ep {ev_epoch[best_i]}, {ev_f[best_i]:.3f})")
    ax.set_xlabel("epoch"); ax.set_ylabel("score"); ax.set_title("eval metrics")
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)

    ax = axes[2]
    labels = counts["labels"]
    x = list(range(len(labels)))
    w = 0.35
    ax.bar([i - w / 2 for i in x], counts["true"], width=w, label="true")
    ax.bar([i + w / 2 for i in x], counts["pred"], width=w, label="pred")
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylabel("tokens"); ax.set_title("val O/B/I counts")
    ax.legend()

    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
names = list(best_f1_by_class)
vals = [best_f1_by_class[n] for n in names]
ax.bar(names, vals)
ax.set_ylim(0, 1); ax.set_ylabel("best eval F1"); ax.set_title("best F1 by class")
for n, v in zip(names, vals):
    ax.text(n, v + 0.02, f"{v:.3f}", ha="center")
plt.tight_layout()
plt.show()

## Prediction viewer
Highlights predicted spans in a paragraph. If a gold file is present at `input/gold.ann`, shows gold vs predicted stacked.

In [ ]:
import html
import os
from IPython.display import HTML, display

INPUT = "input/chap_1_&_2.txt"
PRED = "predictions.ann"
GOLD = "input/gold.ann"  # optional

COLORS = {
    "Deontic": "#ffd59e",
    "DirectObject": "#a8d8ea",
    "Attribute": "#c3e8b7",
}


def parse_ann(path):
    out = []
    with open(path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            meta = parts[1].split(" ")
            cls, s, e = meta[0], int(meta[1]), int(meta[2])
            out.append((cls, s, e, parts[2]))
    return out


def paragraph_bounds(text):
    bounds = []
    i = 0
    while i < len(text):
        while i < len(text) and text[i] == "\n":
            i += 1
        start = i
        while i < len(text) and text[i] != "\n":
            i += 1
        if start < i:
            bounds.append((start, i))
    return bounds


def render_paragraph(text, pstart, pend, spans):
    membership = [frozenset() for _ in range(pend - pstart)]
    for cls, s, e, _ in spans:
        if e <= pstart or s >= pend:
            continue
        a = max(s, pstart) - pstart
        b = min(e, pend) - pstart
        for i in range(a, b):
            membership[i] = membership[i] | {cls}
    out = []
    i = 0
    while i < len(membership):
        cur = membership[i]
        j = i
        while j < len(membership) and membership[j] == cur:
            j += 1
        chunk = html.escape(text[pstart + i:pstart + j])
        if cur:
            if len(cur) == 1:
                bg = f"background:{COLORS[next(iter(cur))]}"
            else:
                items = sorted(cur)
                stops = []
                for k, c in enumerate(items):
                    stops.append(f"{COLORS[c]} {100*k/len(items)}%")
                    stops.append(f"{COLORS[c]} {100*(k+1)/len(items)}%")
                bg = f"background:linear-gradient(90deg,{','.join(stops)})"
            style = f"{bg};padding:0 2px;border-radius:3px"
            title = ", ".join(sorted(cur))
            out.append(f'<span style="{style}" title="{html.escape(title)}">{chunk}</span>')
        else:
            out.append(chunk)
        i = j
    return "".join(out)


def pick_paragraph(bounds, spans, paragraph_index=None):
    if paragraph_index is not None:
        return paragraph_index
    for idx, (s, e) in enumerate(bounds):
        for _, S, E, _ in spans:
            if S < e and E > s:
                return idx
    return 0


def show(paragraph_index=None):
    with open(INPUT) as f:
        text = f.read()
    pred = parse_ann(PRED)
    gold = parse_ann(GOLD) if os.path.exists(GOLD) else None

    # assert text[start:end] == span text for everything we display
    for cls, s, e, t in pred:
        assert text[s:e] == t, f"pred mismatch at {s}-{e}: {text[s:e]!r} vs {t!r}"
    if gold:
        for cls, s, e, t in gold:
            assert text[s:e] == t, f"gold mismatch at {s}-{e}: {text[s:e]!r} vs {t!r}"

    bounds = paragraph_bounds(text)
    idx = pick_paragraph(bounds, pred, paragraph_index)
    pstart, pend = bounds[idx]

    legend = " ".join(
        f'<span style="background:{c};padding:0 6px;border-radius:3px;margin-right:6px">{n}</span>'
        for n, c in COLORS.items()
    )
    parts = [f"<h3 style='margin:4px 0'>paragraph #{idx}</h3>",
             f"<div style='margin-bottom:8px'>legend: {legend}</div>"]
    if gold is not None:
        parts.append("<h4 style='margin:8px 0 2px'>gold</h4>")
        parts.append(f"<div style='line-height:1.6'>{render_paragraph(text, pstart, pend, gold)}</div>")
        parts.append("<h4 style='margin:8px 0 2px'>predicted</h4>")
    parts.append(f"<div style='line-height:1.6'>{render_paragraph(text, pstart, pend, pred)}</div>")
    display(HTML("".join(parts)))


show()  # pass paragraph_index=N to pick a different one